# Multi-Model Embedding Analysis
This notebook performs a memory-efficient comparative analysis of all organizational embedding models. 

In [ ]:
import numpy as np
import pandas as pd
import os
import gc
import time
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from orgpackage.aux import load_dataset, load_embeddings
from orgpackage.config import DOMAIN_CLASSES_CORR

import warnings
warnings.filterwarnings('ignore')

# Global Plotting Settings
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False
})

os.makedirs("figures", exist_ok=True)

MODELS = ['e5-small', 'multilingual-e5', 'qwen', 'mistral', 'finetuned-me5']

exp_classes = [c for sublist in DOMAIN_CLASSES_CORR.values() for c in sublist]
exp_classes = sorted(list(set(exp_classes)), key=len, reverse=True)

## Metric and Utility Functions

In [ ]:
def compute_metrics(X, y, k=10):
    # 1. FDR (on target classes only)
    mask = y != "Others"
    X_f, y_f = X[mask], y[mask]
    
    fdr = np.nan
    if len(np.unique(y_f)) > 1:
        n_features = X.shape[1]
        global_mean = np.mean(X_f, axis=0)
        S_W = np.zeros((n_features, n_features))
        S_B = np.zeros((n_features, n_features))
        for c in np.unique(y_f):
            X_c = X_f[y_f == c]
            if len(X_c) <= 1: continue
            mean_c = np.mean(X_c, axis=0)
            S_W += (X_c - mean_c).T @ (X_c - mean_c)
            diff = (mean_c - global_mean).reshape(-1, 1)
            S_B += len(X_c) * (diff @ diff.T)
        fdr = np.trace(np.linalg.pinv(S_W) @ S_B)

    # 2. Isotropy
    X_c = X - np.mean(X, axis=0)
    cov = np.cov(X_c, rowvar=False)
    evs = np.linalg.eigvalsh(cov)
    isotropy = evs.min() / evs.max() if evs.max() > 0 else 0

    # 3. IR Metrics (MRR & Recall@k)
    sims = cosine_similarity(X_f)
    np.fill_diagonal(sims, -np.inf)
    mrr_sum = 0
    r_at_k_sum = 0
    for i in range(len(X_f)):
        matches = (y_f[np.argsort(sims[i])[::-1]] == y_f[i])
        if np.any(matches):
            idx = np.argmax(matches)
            mrr_sum += 1.0 / (idx + 1)
            if idx < k: r_at_k_sum += 1

    # 4. Recall Curve Data
    recalls = []
    for rk in range(1, 21):
        hits = 0
        for i in range(len(X_f)):
            matches = (y_f[np.argsort(sims[i])[::-1]] == y_f[i])
            if np.any(matches) and np.argmax(matches) < rk: hits += 1
        recalls.append(hits / len(X_f))

    return float(np.real(fdr)), isotropy, mrr_sum/len(X_f), r_at_k_sum/len(X_f), recalls

def get_target_label(row):
    for c in exp_classes:
        if row.get(c) == 1: return c
    return "Others"

## Sequential Processing Loop
We process one model at a time and clear memory immediately after each iteration.

In [ ]:
full_df = load_dataset()
results = {}

for model in MODELS:
    start_time = time.time()
    print(f"\n{'='*50}")
    print(f"PROCESSING MODEL: {model}")
    print(f"{'='*50}")
    
    path = f"./results/embeddings/{model}_embeddings.csv"
    if not os.path.exists(path):
        print(f"[SKIP] File not found: {path}")
        continue

    print(f"[1/4] Loading large embeddings file... ({os.path.getsize(path)/1e9:.2f} GB)")
    emb_df = load_embeddings(path)
    embedding_col = [col for col in emb_df.columns if col.endswith('_embedding')][0]
    
    print("[2/4] Merging with ground truth and labels...")
    m_df = full_df.merge(emb_df[['instance', embedding_col]], on='instance', how='inner')
    m_df = m_df[m_df[embedding_col].notna()].reset_index(drop=True)
    
    X = np.stack(m_df[embedding_col].values)
    m_df['target_label'] = m_df.apply(get_target_label, axis=1)
    y = m_df['target_label'].values
    
    print("[3/4] calculating metrics and recall curves...")
    fdr, iso, mrr, r10, recall_history = compute_metrics(X, y, k=10)
    
    print("[4/4] Performing PCA for visualization sampling...")
    n_sample = 3000
    idx = np.random.choice(len(X), min(len(X), n_sample), replace=False)
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X[idx])
    
    results[model] = {
        'metrics': {'FDR': fdr, 'Isotropy': iso, 'MRR': mrr, 'Recall@10': r10},
        'recalls': recall_history,
        'pca_points': X_2d,
        'pca_labels': y[idx],
        'pca_var': pca.explained_variance_ratio_
    }
    
    print(f"Done in {time.time() - start_time:.1f}s. Clearing memory...")
    del emb_df, m_df, X, y, X_2d
    gc.collect()

## 1. Comparative Metrics Table

In [ ]:
metrics_df = pd.DataFrame({m: results[m]['metrics'] for m in results}).T
metrics_df.index.name = "Model"
display(metrics_df)
metrics_df.to_csv("results/comparative_embedding_metrics.csv")

## 2. Multi-Model Recall Comparison

In [ ]:
plt.figure(figsize=(10, 6))
for model in results:
    plt.plot(range(1, 21), results[model]['recalls'], label=model, marker='.', markersize=4)

plt.xlabel('k')
plt.ylabel('Recall')
plt.xticks(range(1, 21, 2))
plt.ylim(0, 1.05)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig("figures/comparative_recall_all_models.pdf")
plt.show()

## 3. PCA Subplot Comparison

In [ ]:
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 5), sharex=True, sharey=True)
if n_models == 1: axes = [axes]

cmap = plt.get_cmap('tab10')
target_labels = sorted([l for l in exp_classes])

for i, model in enumerate(results):
    ax = axes[i]
    X_2d = results[model]['pca_points']
    y_v = results[model]['pca_labels']
    
    # Plot Others
    mask_o = y_v == "Others"
    ax.scatter(X_2d[mask_o, 0], X_2d[mask_o, 1], c='#f0f0f0', s=10, alpha=0.3, zorder=1)
    
    # Plot Exp Classes
    for j, lab in enumerate(target_labels):
        mask = y_v == lab
        if np.any(mask):
            ax.scatter(X_2d[mask, 0], X_2d[mask, 1], label=lab if i==0 else "", 
                       s=20, alpha=0.9, color=cmap(j % 10), zorder=2)
    
    ax.set_title(model)
    ax.set_xlabel(f"PC1 ({results[model]['pca_var'][0]:.1%})")
    if i == 0: ax.set_ylabel(f"PC2 ({results[model]['pca_var'][1]:.1%})")

fig.legend(bbox_to_anchor=(1.01, 0.5), loc='center left', frameon=False)
plt.tight_layout()
plt.savefig("figures/comparative_pca_subplots.pdf", bbox_inches='tight')
plt.show()